# SRQ4 — ViT Baseline Comparison

Trains EfficientFormer-L1 under the same two-phase transfer learning protocol used for all CNN experiments.
Phase 1 hyperparameters are fixed (identical to CNN baseline). Phase 2 LR and weight decay are selected
via a 3-fold grid search on EfficientFormer before 5-fold final training — ensuring fair per-architecture
tuning without inflating the CNN comparison.

**Protocol notes:**
- Phase 1 `param_mode='head'`: transformer encoder frozen, only classifier head warms up
- Phase 2 `param_mode='all'`: full fine-tuning with gradient clipping (max_norm=1.0)
- Phase 2 patience increased to 5 (vs 3 for CNNs) to allow ViT stabilisation after unfreezing
- Grid search: `lr_phase2` ∈ {1e-5, 5e-5, 1e-4} × `weight_decay` ∈ {1e-4, 1e-3, 1e-2} — 9 configs × 3 folds = 27 runs
- Final training: 5-fold CV with best config; best fold selected by val F1 for test evaluation
- **Section 8:** Final held-out test set evaluation — run once only after all CV decisions are made


## 1 · Paths & Imports

In [1]:
import sys
from pathlib import Path

ABSOLUTE_PATH = Path().resolve()
PROJECT_ROOT  = ABSOLUTE_PATH.parents[1]
DATA_DIR      = PROJECT_ROOT / "data" / "raw"
RESULTS_DIR   = PROJECT_ROOT / "data" / "experiments" / "vit-comparison-results"
WEIGHTS_DIR   = RESULTS_DIR / "weights"
CURVES_DIR    = RESULTS_DIR / "training-curves"
PLOTS_DIR     = RESULTS_DIR / "plots"

# Upstream experiment directories (read-only)
ARCH_EVAL_RESULTS_DIR     = PROJECT_ROOT / "data" / "experiments" / "arch-eval-results"
HEAD_ABLATION_RESULTS_DIR = PROJECT_ROOT / "data" / "experiments" / "head-ablation-results"
GRID_SEARCH_RESULTS_DIR   = PROJECT_ROOT / "data" / "experiments" / "grid-search-results"

for d in [RESULTS_DIR, WEIGHTS_DIR, CURVES_DIR, PLOTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

sys.path.append(str(PROJECT_ROOT))

print(f"Project root : {PROJECT_ROOT}")
print(f"Data dir     : {DATA_DIR}")
print(f"Results dir  : {RESULTS_DIR}")
print(f"Weights dir  : {WEIGHTS_DIR}")

Project root : C:\Users\markm\Workspace\ms-machine-learning-diagnosis
Data dir     : C:\Users\markm\Workspace\ms-machine-learning-diagnosis\data\raw
Results dir  : C:\Users\markm\Workspace\ms-machine-learning-diagnosis\data\experiments\vit-comparison-results
Weights dir  : C:\Users\markm\Workspace\ms-machine-learning-diagnosis\data\experiments\vit-comparison-results\weights


In [2]:
import csv
import time
import traceback
from datetime import datetime
from itertools import product

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from sklearn.metrics import f1_score

import src.scripts.data      as data
import src.scripts.models    as models
import src.scripts.trainer   as trainer
import src.scripts.utils     as utils
import src.scripts.evaluator as evaluator

utils.set_seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Random seed set to 42 for Python, NumPy, and PyTorch
Device: cpu


## 2 · Data — Outer Split (Identical to All Other Experiments)

Fixed seed 42 outer 80/20 stratified split. All training and model selection operates within `X_trainval`.
The held-out test set is set aside until final evaluation in Section 8.

In [3]:
path, categories = data.get_dataset(str(DATA_DIR))
classes = data.get_classes(path, categories, visualise=False)

image_paths, labels = data.get_paths_and_labels(path, classes)
train_transform, test_transform = data.get_transforms()

X_trainval, y_trainval, X_test, y_test = data.get_trainval_test_split(
    image_paths, labels,
    test_split=0.20,
    SEED=42
)

print(f"\nSRQ4 operates on {len(X_trainval)} train+val samples.")
print("Held-out test set is NOT used until Section 8 (final evaluation).")

get_dataset()>>> Dataset already exists in C:\Users\markm\Workspace\ms-machine-learning-diagnosis\data\raw
get_dataset()>>> Available categories: ['Control Axial_crop', 'Control Saggital_crop', 'MS Axial_crop', 'MS Saggital_crop']
get_paths_and_labels()>>> Total images: 1652
get_trainval_test_split()>>> Train+Val pool : 1321 (80.0%)
get_trainval_test_split()>>> Held-out test  : 331 (20.0%)
get_trainval_test_split()>>> TrainVal class ratio — MS: 520  Non-MS: 801
get_trainval_test_split()>>> Test     class ratio — MS: 130  Non-MS: 201

SRQ4 operates on 1321 train+val samples.
Held-out test set is NOT used until Section 8 (final evaluation).


## 3 · Fixed Configuration

Phase 1 hyperparameters are fixed a priori — identical to all CNN experiments.
Phase 2 LR and weight decay are selected via grid search in Section 4.
The linear head is fixed to match all other experiments (established by head ablation).

In [4]:
# ── Shared fixed parameters — identical to all CNN experiments ────────────────
SEED        = 42
BATCH_SIZE  = 32
WINNING_HEAD = "linear"   # established by head ablation; fixed for all experiments

# Phase 1 — fixed a priori (same as CNN)
LR_PHASE1   = 1e-3
WD_PHASE1   = 0.0
P1_EPOCHS   = 20
P1_PATIENCE = 4

# Phase 2 — epoch ceiling and patience fixed; LR/WD selected by grid search
# Patience increased to 5 vs CNN's 3: ViTs take longer to stabilise after unfreezing
P2_EPOCHS      = 15
P2_PATIENCE    = 5
GRAD_CLIP_NORM = 1.0   # gradient clipping — standard practice for ViT fine-tuning

ARCHITECTURE = "efficientformer"

print(f"Architecture  : {ARCHITECTURE}")
print(f"Head          : {WINNING_HEAD}")
print(f"Phase 1       : lr={LR_PHASE1:.0e}  wd={WD_PHASE1}  epochs={P1_EPOCHS}  patience={P1_PATIENCE}")
print(f"Phase 2       : epochs={P2_EPOCHS}  patience={P2_PATIENCE}  grad_clip={GRAD_CLIP_NORM}")
print(f"               (LR and WD selected by grid search — see Section 4)")

Architecture  : efficientformer
Head          : linear
Phase 1       : lr=1e-03  wd=0.0  epochs=20  patience=4
Phase 2       : epochs=15  patience=5  grad_clip=1.0
               (LR and WD selected by grid search — see Section 4)


## 4 · Phase 2 Grid Search (3-Fold CV)

Searches `lr_phase2` × `weight_decay` using 3-fold CV on `X_trainval`.
The same grid search procedure used for ResNet18 — applied here with a ViT-appropriate LR range.
Results are saved fold-by-fold; safe to interrupt and resume.

**Grid:** `lr_phase2` ∈ {1e-5, 5e-5, 1e-4} × `weight_decay` ∈ {1e-4, 1e-3, 1e-2} = 9 configs × 3 folds = 27 runs

In [5]:
# ── Grid search space ─────────────────────────────────────────────────────────
LR_PHASE2_OPTIONS    = [1e-5, 5e-5, 1e-4]
WEIGHT_DECAY_OPTIONS = [1e-4, 1e-3, 1e-2]
GRID_SEARCH_SPLITS   = 3

GS_RESULTS_FILE = RESULTS_DIR / "grid_search_results.csv"
GS_FIELDNAMES = [
    "lr_phase2", "weight_decay", "fold",
    "val_f1", "val_acc", "val_loss",
    "p1_epochs_run", "p2_epochs_run",
    "timestamp", "error"
]

configs = list(product(LR_PHASE2_OPTIONS, WEIGHT_DECAY_OPTIONS))
total_gs_runs = len(configs) * GRID_SEARCH_SPLITS
print(f"Grid: {len(LR_PHASE2_OPTIONS)} LRs × {len(WEIGHT_DECAY_OPTIONS)} WDs "
      f"× {GRID_SEARCH_SPLITS} folds = {total_gs_runs} runs")
print(f"Results → {GS_RESULTS_FILE}")

Grid: 3 LRs × 3 WDs × 3 folds = 27 runs
Results → C:\Users\markm\Workspace\ms-machine-learning-diagnosis\data\experiments\vit-comparison-results\grid_search_results.csv


In [ ]:
completed_gs = utils.load_completed_runs(
    GS_RESULTS_FILE,
    [("lr_phase2", float), ("weight_decay", float), ("fold", int)]
)
gs_run_number = len(completed_gs)

for lr_p2, wd in configs:
    for fold_idx in range(GRID_SEARCH_SPLITS):

        run_key = (lr_p2, wd, fold_idx)
        if run_key in completed_gs:
            print(f"SKIP  lr={lr_p2:.0e}  wd={wd:.0e}  fold={fold_idx+1}/{GRID_SEARCH_SPLITS}")
            continue

        utils.set_seed(SEED)
        gs_run_number += 1
        print(f"\n{'='*65}")
        print(f"  GS Run {gs_run_number}/{total_gs_runs}  |  "
              f"lr_p2={lr_p2:.0e}  wd={wd:.0e}  fold={fold_idx+1}/{GRID_SEARCH_SPLITS}")
        print(f"{'='*65}")

        error_msg = ""
        val_f1 = val_acc = val_loss = float("nan")
        p1_epochs_run = p2_epochs_run = 0

        try:
            train_loader, val_loader = data.get_fold_loaders(
                X_trainval, y_trainval,
                fold_idx=fold_idx,
                train_transform=train_transform,
                test_transform=test_transform,
                n_splits=GRID_SEARCH_SPLITS,
                batch_size=BATCH_SIZE,
                SEED=SEED
            )

            model = models.get_model(architecture=ARCHITECTURE, head=WINNING_HEAD)

            run_configs = {
                "phase1": {
                    "num_epochs"  : P1_EPOCHS,
                    "lr"          : LR_PHASE1,
                    "parameters"  : "head",
                    "optimiser"   : optim.AdamW,
                    "criterion"   : nn.BCEWithLogitsLoss(),
                    "weight_decay": WD_PHASE1,
                },
                "phase2": {
                    "num_epochs"  : P2_EPOCHS,
                    "lr"          : lr_p2,
                    "parameters"  : "all",
                    "optimiser"   : optim.AdamW,
                    "criterion"   : nn.BCEWithLogitsLoss(),
                    "weight_decay": wd,
                },
            }

            # Phase 1
            losses_p1, accs_p1, val_losses_p1, val_accs_p1 = trainer.train_model(
                model, train_loader, val_loader,
                config_name="phase1",
                train_configs=run_configs,
                early_stopping_patience=P1_PATIENCE
            )
            p1_epochs_run = len(val_accs_p1)

            # Phase 2 with gradient clipping
            # Note: trainer.train_model does not apply grad clipping natively;
            # we patch it in via a thin wrapper that clips after loss.backward().
            losses_p2, accs_p2, val_losses_p2, val_accs_p2 = trainer.train_model(
                model, train_loader, val_loader,
                config_name="phase2",
                train_configs=run_configs,
                early_stopping_patience=P2_PATIENCE,
                grad_clip_norm=GRAD_CLIP_NORM
            )
            p2_epochs_run = len(val_accs_p2)

            # Val metrics from final epoch
            val_loss = val_losses_p2[-1]
            val_acc  = val_accs_p2[-1]

            # Compute val F1 on validation set
            model.eval()
            y_true_val, y_pred_val = [], []
            with torch.no_grad():
                for imgs, lbls in val_loader:
                    imgs = imgs.to(DEVICE)
                    probs = torch.sigmoid(model(imgs)).cpu().numpy().flatten()
                    preds = (probs >= 0.5).astype(int)
                    y_true_val.extend(lbls.numpy().astype(int))
                    y_pred_val.extend(preds)
            val_f1 = f1_score(y_true_val, y_pred_val)

            print(f"  val_f1={val_f1:.4f}  val_acc={val_acc:.4f}  val_loss={val_loss:.4f}  "
                  f"(p1={p1_epochs_run}ep  p2={p2_epochs_run}ep)")

        except Exception as e:
            error_msg = str(e)
            print(f"  ERROR: {error_msg}")
            traceback.print_exc()

        utils.append_result(GS_RESULTS_FILE, GS_FIELDNAMES, {
            "lr_phase2"    : lr_p2,
            "weight_decay" : wd,
            "fold"         : fold_idx,
            "val_f1"       : round(val_f1, 4) if not np.isnan(val_f1) else "",
            "val_acc"      : round(val_acc, 4) if not np.isnan(val_acc) else "",
            "val_loss"     : round(val_loss, 4) if not np.isnan(val_loss) else "",
            "p1_epochs_run": p1_epochs_run,
            "p2_epochs_run": p2_epochs_run,
            "timestamp"    : datetime.now().isoformat(timespec="seconds"),
            "error"        : error_msg,
        })

print(f"\n{'='*65}")
print("GRID SEARCH COMPLETE")
print(f"Results → {GS_RESULTS_FILE}")

Random seed set to 42 for Python, NumPy, and PyTorch

  GS Run 1/27  |  lr_p2=1e-05  wd=1e-04  fold=1/3
get_fold_loaders()>>> Fold 1/3 — Train: 880,  Val: 441
get_model()>>> architecture='efficientformer'  head='linear'
[phase1] Epoch 1/20 - Train Loss: 0.6379 - Train Acc: 0.6455 - Val Loss: 0.6009 - Val Acc: 0.6531


## 5 · Grid Search Analysis — Select Best Phase 2 Config

In [ ]:
df_gs = pd.read_csv(GS_RESULTS_FILE)
df_gs = df_gs[df_gs["error"] == ""].copy()
df_gs["val_f1"] = df_gs["val_f1"].astype(float)

print(f"Completed grid search runs: {len(df_gs)} / {total_gs_runs}")

gs_summary = (
    df_gs.groupby(["lr_phase2", "weight_decay"])
    .agg(
        mean_val_f1 = ("val_f1", "mean"),
        std_val_f1  = ("val_f1", "std"),
        n_folds     = ("val_f1", "count"),
    )
    .reset_index()
    .sort_values("mean_val_f1", ascending=False)
)

gs_summary["display"] = gs_summary.apply(
    lambda r: f"{r.mean_val_f1:.4f} ± {r.std_val_f1:.4f}", axis=1
)

print("\nGrid search results (sorted by mean val F1):")
print(gs_summary[["lr_phase2", "weight_decay", "display", "n_folds"]].to_string(index=False))

In [ ]:
# ── Select best config ────────────────────────────────────────────────────────
best_gs = gs_summary.sort_values(
    ["mean_val_f1", "std_val_f1"],
    ascending=[False, True]   # highest mean, then lowest std as tiebreak
).iloc[0]

BEST_LR_PHASE2    = float(best_gs["lr_phase2"])
BEST_WEIGHT_DECAY = float(best_gs["weight_decay"])

print("\n" + "="*60)
print("  SELECTED PHASE 2 CONFIG")
print("="*60)
print(f"  lr_phase2    = {BEST_LR_PHASE2:.0e}")
print(f"  weight_decay = {BEST_WEIGHT_DECAY:.0e}")
print(f"  mean val F1  = {best_gs.mean_val_f1:.4f} ± {best_gs.std_val_f1:.4f}")
print("="*60)

# Save optimal config for reference
optimal_config_path = RESULTS_DIR / "optimal_vit_config.csv"
pd.DataFrame([{
    "lr_phase1"   : LR_PHASE1,
    "wd_phase1"   : WD_PHASE1,
    "lr_phase2"   : BEST_LR_PHASE2,
    "weight_decay": BEST_WEIGHT_DECAY,
}]).to_csv(optimal_config_path, index=False)
print(f"\nConfig saved → {optimal_config_path}")

In [ ]:
# ── Grid search heatmap ───────────────────────────────────────────────────────
pivot = gs_summary.pivot(index="lr_phase2", columns="weight_decay", values="mean_val_f1")

fig, ax = plt.subplots(figsize=(7, 4))
im = ax.imshow(pivot.values, aspect="auto", cmap="YlGn")
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([f"{v:.0e}" for v in pivot.columns], fontsize=9)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels([f"{v:.0e}" for v in pivot.index], fontsize=9)
ax.set_xlabel("Weight Decay", fontsize=10)
ax.set_ylabel("LR Phase 2", fontsize=10)
ax.set_title("EfficientFormer — Grid Search Mean Val F1 (3-fold)", fontsize=11)
plt.colorbar(im, ax=ax)
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        v = pivot.values[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.4f}", ha="center", va="center", fontsize=8,
                    color="white" if v > pivot.values.mean() else "black")
plt.tight_layout()
utils.save_fig(fig, PLOTS_DIR, "grid_search_heatmap", formats=("svg",))
plt.show()

## 6 · 5-Fold CV Training — EfficientFormer

Trains EfficientFormer under the selected Phase 2 config using 5-fold stratified CV.
Weights saved per fold. Results saved fold-by-fold; safe to interrupt and resume.

In [ ]:
N_SPLITS     = 5
RESULTS_FILE = RESULTS_DIR / "vit_cv_results.csv"
CV_FIELDNAMES = [
    "architecture", "lr_phase1", "wd_phase1", "lr_phase2", "weight_decay",
    "fold", "val_acc", "val_loss", "val_f1",
    "p1_epochs_run", "p2_epochs_run",
    "weights_path", "timestamp", "error"
]

total_cv_runs = N_SPLITS
print(f"EfficientFormer × {N_SPLITS} folds = {total_cv_runs} runs")
print(f"lr_phase2 : {BEST_LR_PHASE2:.0e}  weight_decay : {BEST_WEIGHT_DECAY:.0e}")
print(f"Results   → {RESULTS_FILE}")
print(f"Weights   → {WEIGHTS_DIR}/{ARCHITECTURE}/fold_<n>.pt")

In [ ]:
completed_cv = utils.load_completed_runs(
    RESULTS_FILE,
    [("architecture", str), ("fold", int)]
)
cv_run_number = len(completed_cv)
print(f"Device: {DEVICE}")
print(f"{N_SPLITS} folds — {cv_run_number} already completed\n")

for fold_idx in range(N_SPLITS):

    run_key = (ARCHITECTURE, fold_idx)
    if run_key in completed_cv:
        print(f"SKIP  fold={fold_idx+1}/{N_SPLITS}")
        continue

    utils.set_seed(SEED)
    cv_run_number += 1
    print(f"\n{'='*65}")
    print(f"  Run {cv_run_number}/{total_cv_runs}  |  fold={fold_idx+1}/{N_SPLITS}")
    print(f"{'='*65}")

    error_msg = ""
    val_f1 = val_acc = val_loss = float("nan")
    p1_epochs_run = p2_epochs_run = 0
    wpath = utils.weights_path_for(WEIGHTS_DIR, ARCHITECTURE, fold_idx)

    try:
        train_loader, val_loader = data.get_fold_loaders(
            X_trainval, y_trainval,
            fold_idx=fold_idx,
            train_transform=train_transform,
            test_transform=test_transform,
            n_splits=N_SPLITS,
            batch_size=BATCH_SIZE,
            SEED=SEED
        )

        model = models.get_model(architecture=ARCHITECTURE, head=WINNING_HEAD)

        run_configs = {
            "phase1": {
                "num_epochs"  : P1_EPOCHS,
                "lr"          : LR_PHASE1,
                "parameters"  : "head",
                "optimiser"   : optim.AdamW,
                "criterion"   : nn.BCEWithLogitsLoss(),
                "weight_decay": WD_PHASE1,
            },
            "phase2": {
                "num_epochs"  : P2_EPOCHS,
                "lr"          : BEST_LR_PHASE2,
                "parameters"  : "all",
                "optimiser"   : optim.AdamW,
                "criterion"   : nn.BCEWithLogitsLoss(),
                "weight_decay": BEST_WEIGHT_DECAY,
            },
        }

        # Phase 1
        losses_p1, accs_p1, val_losses_p1, val_accs_p1 = trainer.train_model(
            model, train_loader, val_loader,
            config_name="phase1",
            train_configs=run_configs,
            early_stopping_patience=P1_PATIENCE
        )
        p1_epochs_run = len(val_accs_p1)

        utils.plot(
            losses_p1, accs_p1,
            config_name=f"phase1_fold{fold_idx}",
            val_losses=val_losses_p1, val_accuracies=val_accs_p1,
            model_name=ARCHITECTURE,
            save_dir=str(CURVES_DIR),
            show=False
        )

        # Phase 2 with gradient clipping
        losses_p2, accs_p2, val_losses_p2, val_accs_p2 = trainer.train_model(
            model, train_loader, val_loader,
            config_name="phase2",
            train_configs=run_configs,
            early_stopping_patience=P2_PATIENCE,
            grad_clip_norm=GRAD_CLIP_NORM
        )
        p2_epochs_run = len(val_accs_p2)

        utils.plot(
            losses_p2, accs_p2,
            config_name=f"phase2_fold{fold_idx}",
            val_losses=val_losses_p2, val_accuracies=val_accs_p2,
            model_name=ARCHITECTURE,
            save_dir=str(CURVES_DIR),
            show=False
        )

        val_loss = val_losses_p2[-1]
        val_acc  = val_accs_p2[-1]

        # Val F1
        model.eval()
        y_true_val, y_pred_val = [], []
        with torch.no_grad():
            for imgs, lbls in val_loader:
                imgs = imgs.to(DEVICE)
                probs = torch.sigmoid(model(imgs)).cpu().numpy().flatten()
                preds = (probs >= 0.5).astype(int)
                y_true_val.extend(lbls.numpy().astype(int))
                y_pred_val.extend(preds)
        val_f1 = f1_score(y_true_val, y_pred_val)

        utils.save_weights(model, wpath)
        print(f"  val_f1={val_f1:.4f}  val_acc={val_acc:.4f}  val_loss={val_loss:.4f}  "
              f"(p1={p1_epochs_run}ep  p2={p2_epochs_run}ep)")
        print(f"  Weights → {wpath}")

    except Exception as e:
        error_msg = str(e)
        print(f"  ERROR: {error_msg}")
        traceback.print_exc()

    utils.append_result(RESULTS_FILE, CV_FIELDNAMES, {
        "architecture" : ARCHITECTURE,
        "lr_phase1"    : LR_PHASE1,
        "wd_phase1"    : WD_PHASE1,
        "lr_phase2"    : BEST_LR_PHASE2,
        "weight_decay" : BEST_WEIGHT_DECAY,
        "fold"         : fold_idx,
        "val_acc"      : round(val_acc, 4) if not np.isnan(val_acc) else "",
        "val_loss"     : round(val_loss, 4) if not np.isnan(val_loss) else "",
        "val_f1"       : round(val_f1, 4) if not np.isnan(val_f1) else "",
        "p1_epochs_run": p1_epochs_run,
        "p2_epochs_run": p2_epochs_run,
        "weights_path" : str(wpath),
        "timestamp"    : datetime.now().isoformat(timespec="seconds"),
        "error"        : error_msg,
    })

print(f"\n{'='*65}")
print("CV TRAINING COMPLETE")
print(f"Results → {RESULTS_FILE}")

## 7 · CV Results Summary

In [ ]:
df_cv = pd.read_csv(RESULTS_FILE)
df_cv["error"] = df_cv["error"].fillna("")
df_ok   = df_cv[df_cv["error"] == ""].copy()
df_fail = df_cv[df_cv["error"] != ""].copy()

for col in ["val_f1", "val_acc", "val_loss"]:
    df_ok[col] = df_ok[col].astype(float)

print(f"Successful runs : {len(df_ok)} / {total_cv_runs}")
print(f"Failed runs     : {len(df_fail)}")
if len(df_fail):
    print(df_fail[["fold", "error"]].to_string(index=False))

In [ ]:
if not df_ok.empty:
    print(f"EfficientFormer — CV Summary\n")
    print(f"  mean val F1   : {df_ok.val_f1.mean():.4f} ± {df_ok.val_f1.std():.4f}")
    print(f"  mean val acc  : {df_ok.val_acc.mean():.4f} ± {df_ok.val_acc.std():.4f}")
    print(f"  mean val loss : {df_ok.val_loss.mean():.4f} ± {df_ok.val_loss.std():.4f}")
    print()
    print(f"  {'Fold':<6} {'val_f1':>8} {'val_acc':>9} {'val_loss':>10} {'p1_ep':>6} {'p2_ep':>6}")
    print(f"  {'-'*50}")
    for _, r in df_ok.sort_values("fold").iterrows():
        print(f"  {int(r.fold)+1:<6} {r.val_f1:>8.4f} {r.val_acc:>9.4f} "
              f"{r.val_loss:>10.4f} {int(r.p1_epochs_run):>6} {int(r.p2_epochs_run):>6}")

    # Load CNN baseline CV mean for comparison
    arch_eval_csv = ARCH_EVAL_RESULTS_DIR / "arch_eval_results.csv"
    if arch_eval_csv.exists():
        df_arch = pd.read_csv(arch_eval_csv)
        df_arch = df_arch[df_arch["error"] == ""]
        best_cnn_mean = df_arch.groupby("architecture")["val_f1"].mean().max()
        best_cnn_arch = df_arch.groupby("architecture")["val_f1"].mean().idxmax()
        print(f"\n  Best CNN CV mean F1 : {best_cnn_mean:.4f} ({best_cnn_arch})")
        print(f"  ViT CV mean F1      : {df_ok.val_f1.mean():.4f}")
        print(f"  Δ (ViT − CNN)       : {df_ok.val_f1.mean() - best_cnn_mean:+.4f}")

In [ ]:
# ── Fold selection — best val F1 ──────────────────────────────────────────────
if not df_ok.empty:
    best_fold_row = df_ok.sort_values("val_f1", ascending=False).iloc[0]
    SELECTED_FOLD = int(best_fold_row["fold"])
    print(f"Selected fold {SELECTED_FOLD+1}  "
          f"(val_f1={best_fold_row.val_f1:.4f}  val_loss={best_fold_row.val_loss:.4f})")

## 8 · Final Test Set Evaluation — SRQ4

Loads the best fold weights for EfficientFormer and the best CNN from arch-eval.
Both evaluated once on the held-out test set. Run once only after all model selection decisions are made.

In [ ]:
test_loader = data.get_test_loader(X_test, y_test, test_transform, batch_size=BATCH_SIZE)
print(f"Device: {DEVICE}")

In [ ]:
test_results = {}

# ── Evaluate EfficientFormer ──────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"  Test evaluation: {ARCHITECTURE}  (fold {SELECTED_FOLD+1})")
print(f"{'='*60}")

vit_weights = utils.weights_path_for(WEIGHTS_DIR, ARCHITECTURE, SELECTED_FOLD)

if not vit_weights.exists():
    print(f"  SKIP — weights not found at {vit_weights}")
else:
    utils.set_seed(SEED)
    vit_model = models.get_model(architecture=ARCHITECTURE, head=WINNING_HEAD)
    vit_model = utils.load_weights(vit_model, vit_weights, device=DEVICE)

    acc, prec, rec, f1, auc, ece, conf, report = evaluator.evaluate_model(
        model=vit_model, test_loader=test_loader, device=DEVICE
    )
    test_results[ARCHITECTURE] = {
        "test_f1": round(f1, 4), "test_auc": round(auc, 4),
        "test_ece": round(ece, 4), "test_acc": round(acc, 4),
        "test_prec": round(prec, 4), "test_rec": round(rec, 4),
        "conf_matrix": conf.tolist(),
        "lr_phase2": BEST_LR_PHASE2, "weight_decay": BEST_WEIGHT_DECAY,
    }

In [ ]:
# ── Evaluate best CNN from arch-eval ─────────────────────────────────────────
arch_eval_csv = ARCH_EVAL_RESULTS_DIR / "arch_eval_results.csv"

if arch_eval_csv.exists():
    df_arch = pd.read_csv(arch_eval_csv)
    df_arch = df_arch[df_arch["error"] == ""]

    # Best CNN arch by mean val F1
    best_cnn_arch = df_arch.groupby("architecture")["val_f1"].mean().idxmax()
    # Best fold for that arch
    cnn_best_fold = int(
        df_arch[df_arch["architecture"] == best_cnn_arch]
        .sort_values("val_f1", ascending=False)
        .iloc[0]["fold"]
    )
    cnn_weights = ARCH_EVAL_RESULTS_DIR / "weights" / best_cnn_arch / f"fold_{cnn_best_fold}.pt"

    print(f"\n{'='*60}")
    print(f"  Test evaluation: {best_cnn_arch} (best CNN, fold {cnn_best_fold+1})")
    print(f"{'='*60}")

    if not cnn_weights.exists():
        print(f"  SKIP — weights not found at {cnn_weights}")
    else:
        utils.set_seed(SEED)
        cnn_model = models.get_model(architecture=best_cnn_arch, head=WINNING_HEAD)
        cnn_model = utils.load_weights(cnn_model, cnn_weights, device=DEVICE)

        acc, prec, rec, f1, auc, ece, conf, report = evaluator.evaluate_model(
            model=cnn_model, test_loader=test_loader, device=DEVICE
        )
        test_results[best_cnn_arch] = {
            "test_f1": round(f1, 4), "test_auc": round(auc, 4),
            "test_ece": round(ece, 4), "test_acc": round(acc, 4),
            "test_prec": round(prec, 4), "test_rec": round(rec, 4),
            "conf_matrix": conf.tolist(),
            "lr_phase2": None, "weight_decay": None,
        }
else:
    print("arch-eval results not found — CNN comparison skipped")

In [ ]:
# ── SRQ4 comparison table ─────────────────────────────────────────────────────
df_test = pd.DataFrame(test_results).T
df_test = df_test.drop(columns=["conf_matrix"])
for col in ["test_f1", "test_auc", "test_ece", "test_acc", "test_prec", "test_rec"]:
    df_test[col] = df_test[col].astype(float)

print("SRQ4 — Final Test Results\n")
print(f"{'Model':<30} {'F1':>7} {'AUC':>7} {'ECE':>7} {'Acc':>7}")
print("-" * 58)
for arch, row in df_test.iterrows():
    print(f"  {arch:<28} {row.test_f1:>7.4f} {row.test_auc:>7.4f} "
          f"{row.test_ece:>7.4f} {row.test_acc:>7.4f}")

# Delta
if ARCHITECTURE in df_test.index and best_cnn_arch in df_test.index:
    delta_f1  = df_test.loc[ARCHITECTURE, "test_f1"]  - df_test.loc[best_cnn_arch, "test_f1"]
    delta_auc = df_test.loc[ARCHITECTURE, "test_auc"] - df_test.loc[best_cnn_arch, "test_auc"]
    delta_ece = df_test.loc[ARCHITECTURE, "test_ece"] - df_test.loc[best_cnn_arch, "test_ece"]
    print(f"\n  Δ ({ARCHITECTURE} − {best_cnn_arch}):")
    print(f"    ΔF1  = {delta_f1:+.4f}")
    print(f"    ΔAUC = {delta_auc:+.4f}")
    print(f"    ΔECE = {delta_ece:+.4f}  (negative = better calibrated)")

In [ ]:
# ── Save test results ─────────────────────────────────────────────────────────
TEST_RESULTS_FILE = RESULTS_DIR / "srq4_test_results.csv"
df_test.to_csv(TEST_RESULTS_FILE)
print(f"Test results saved → {TEST_RESULTS_FILE}")

In [ ]:
# ── Visualisation ─────────────────────────────────────────────────────────────
if not df_test.empty:
    COLOUR_MAP = {
        ARCHITECTURE: "#4C72B0",   # blue — ViT
    }
    if best_cnn_arch in df_test.index:
        COLOUR_MAP[best_cnn_arch] = "#DD8452"   # amber — CNN

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle("SRQ4 — CNN vs EfficientFormer", fontsize=13, fontweight="bold")

    plot_order = df_test.sort_values("test_f1", ascending=True)
    colours    = [COLOUR_MAP.get(a, "#888888") for a in plot_order.index]
    y_pos      = np.arange(len(plot_order))

    # Panel 1: F1 bar chart
    ax = axes[0]
    ax.barh(y_pos, plot_order["test_f1"], color=colours,
            alpha=0.85, height=0.5, edgecolor="white", linewidth=0.8)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(plot_order.index, fontsize=10)
    ax.set_xlabel("Test F1", fontsize=11)
    ax.set_title("Test F1", fontsize=11, fontweight="bold")
    ax.spines[["top", "right"]].set_visible(False)
    f1_vals = plot_order["test_f1"].values
    pad = max((f1_vals.max() - f1_vals.min()) * 0.4, 0.02)
    ax.set_xlim(max(0, f1_vals.min() - pad), min(1, f1_vals.max() + pad))
    for y, v in zip(y_pos, f1_vals):
        ax.text(v + 0.002, y, f"{v:.4f}", va="center", fontsize=9)

    # Panel 2: AUC and ECE grouped bars
    ax2   = axes[1]
    x     = np.arange(len(df_test))
    w     = 0.35
    ax2.bar(x - w/2, df_test["test_auc"], w, label="AUC-ROC",
            color="#4C72B0", alpha=0.82)
    ax2.bar(x + w/2, df_test["test_ece"], w, label="ECE (↓ better)",
            color="#DD8452", alpha=0.82)
    ax2.set_xticks(x)
    ax2.set_xticklabels(list(df_test.index), rotation=15, ha="right", fontsize=9)
    ax2.set_ylabel("Score", fontsize=11)
    ax2.set_title("AUC-ROC & ECE", fontsize=11, fontweight="bold")
    ax2.legend(fontsize=9)
    ax2.spines[["top", "right"]].set_visible(False)

    plt.tight_layout()
    utils.save_fig(fig, PLOTS_DIR, "srq4_test_results", formats=("svg",))
    plt.show()